In [1]:
[
    import pandas as pd
import numpy as np
from scipy.optimize import minimize
import seaborn as sns
import matplotlib.pyplot as plt

# --- NEW: Load the data from your Task 1 file ---
data = pd.read_csv("../data/cleaned_financial_data.csv", index_col=0, parse_dates=True)



SyntaxError: '[' was never closed (2483097578.py, line 1)

In [2]:
# 1. Setup Data & Custom View
returns = data.pct_change().dropna()
avg_returns = returns.mean() * 252
avg_returns['TSLA'] = 0.45  # Your Custom LSTM View
cov_matrix = returns.cov() * 252

# 2. Define Portfolio Functions
def get_stats(weights):
    weights = np.array(weights)
    ret = np.sum(avg_returns * weights)
    vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    sharpe = ret / vol
    return np.array([ret, vol, sharpe])

# 3. The Optimizer (Max Sharpe)
def min_func_sharpe(weights):
    return -get_stats(weights)[2] 

cons = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1}) 
bounds = tuple((0, 1) for _ in range(3)) 
init_guess = [0.33, 0.33, 0.34]

opt_results = minimize(min_func_sharpe, init_guess, method='SLSQP', bounds=bounds, constraints=cons)
best_weights = opt_results.x

# 4. Results
print("\n--- OPTIMAL PORTFOLIO (MAX SHARPE) ---")
for i, asset in enumerate(avg_returns.index):
    print(f"{asset}: {best_weights[i]:.2%}")
    


NameError: name 'data' is not defined

In [ ]:
# Get performance metrics for the recommendation
final_stats = get_stats(best_weights)
print(f"\nExpected Annual Return: {final_stats[0]:.2%}")
print(f"Expected Volatility: {final_stats[1]:.2%}")
print(f"Sharpe Ratio: {final_stats[2]:.2%}")

# --- DELIVERABLE: Covariance Heatmap ---

plt.figure(figsize=(8, 6))
sns.heatmap(cov_matrix, annot=True, cmap='coolwarm', fmt=".6f")
plt.title("Asset Covariance Heatmap (Task 4)")
plt.show()